
# SkyShield — Model Improvement and Validation

## SWYNEX Internship — Task 3

### Objective
Improve and validate the baseline machine learning model developed in Task 2 using hyperparameter tuning and cross-validation.

### Baseline Results
- Logistic Regression Macro F1-score: 0.2821
- Random Forest Macro F1-score: 0.2555

### Selected Baseline Model
Logistic Regression, selected based on its higher Macro F1-score.


In [4]:
# Import required libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# Load the cleaned aviation wildlife strike dataset
df = pd.read_csv("SkyShield_Cleaned_Data.csv")

# Display dataset dimensions and preview
print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (358433, 20)


,INDEX_NR,INCIDENT_DATE,INCIDENT_YEAR,INCIDENT_MONTH,INCIDENT_WEEKDAY_NAME,TIME_OF_DAY,AIRPORT,STATE,AIRCRAFT,OPERATOR,PHASE_OF_FLIGHT,HEIGHT,SPEED,DISTANCE,SPECIES,NUM_STRUCK,INDICATED_DAMAGE,DAMAGE_LEVEL,COST_REPAIRS,COST_OTHER
0,608242,1996-06-22,1996,6,Saturday,NaN,SACRAMENTO INTL,CA,B-737-300,UNITED AIRLINES,Take-off Run,0.0,NaN,0.0,Unknown bird - medium,1,0,NaN,NaN,NaN
1,608243,1996-06-26,1996,6,Wednesday,NaN,DENVER INTL AIRPORT,CO,B-737-300,UNITED AIRLINES,Take-off Run,0.0,NaN,0.0,Unknown bird - medium,1,0,NaN,NaN,NaN
2,608244,1996-07-01,1996,7,Monday,NaN,EPPLEY AIRFIELD,NE,B-757-200,UNITED AIRLINES,Take-off Run,0.0,NaN,0.0,Unknown bird - medium,1,0,N,NaN,NaN
3,608245,1996-07-01,1996,7,Monday,NaN,WASHINGTON DULLES INTL ARPT,DC,A-320,UNITED AIRLINES,Approach,1000.0,NaN,NaN,Unknown bird - medium,1,0,N,NaN,NaN
4,608246,1996-07-01,1996,7,Monday,NaN,LA GUARDIA ARPT,NY,A-320,UNITED AIRLINES,Climb,5000.0,NaN,NaN,Unknown bird - medium,1,1,M,NaN,NaN


In [5]:
# Remove rows where the target variable is missing
model_data = df.dropna(subset=["DAMAGE_LEVEL"]).copy()

# Separate features and target
X = model_data.drop(
    columns=[
        "INDEX_NR",
        "INCIDENT_DATE",
        "INDICATED_DAMAGE",
        "COST_REPAIRS",
        "COST_OTHER",
        "DAMAGE_LEVEL"
    ],
    errors="ignore"
)

y = model_data["DAMAGE_LEVEL"]

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Identify numerical and categorical columns
numeric_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

# Display dataset split details
print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)
print("Numerical features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Training data shape: (185328, 14)
Testing data shape: (46332, 14)
Numerical features: 5
Categorical features: 9


C:\Users\vanda\AppData\Local\Temp\ipykernel_12068\2851388600.py:33: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(


In [6]:
# Import model tuning and validation tools
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Preprocess numerical and categorical features
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing steps
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# Build a pipeline to prevent data leakage during cross-validation
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1500,
        random_state=42
    ))
])

# Define hyperparameter combinations
param_grid = {
    "classifier__C": [0.1, 1.0, 3.0],
    "classifier__class_weight": ["balanced"]
}

# Use stratified cross-validation and Macro F1 scoring
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=2
)

# Train and compare parameter combinations using training data only
grid_search.fit(X_train, y_train)

# Display the best parameters and cross-validation score
print("Best Parameters:", grid_search.best_params_)
print("Best Cross-Validation Macro F1:", round(grid_search.best_score_, 4))

Fitting 3 folds for each of 3 candidates, totalling 9 fits
Best Parameters: {'classifier__C': 0.1, 'classifier__class_weight': 'balanced'}
Best Cross-Validation Macro F1: 0.2862


In [8]:
# Get predictions from the tuned model
best_model = grid_search.best_estimator_
y_pred_tuned = best_model.predict(X_test)

# Calculate evaluation metrics
tuned_accuracy = accuracy_score(y_test, y_pred_tuned)

tuned_precision = precision_score(
    y_test, y_pred_tuned,
    average="macro", zero_division=0
)

tuned_recall = recall_score(
    y_test, y_pred_tuned,
    average="macro", zero_division=0
)

tuned_f1 = f1_score(
    y_test, y_pred_tuned,
    average="macro", zero_division=0
)

# Display results
print("Accuracy:", round(tuned_accuracy, 4))
print("Macro Precision:", round(tuned_precision, 4))
print("Macro Recall:", round(tuned_recall, 4))
print("Macro F1-score:", round(tuned_f1, 4))

Accuracy: 0.6622
Macro Precision: 0.2723
Macro Recall: 0.4527
Macro F1-score: 0.2837


In [9]:
# Store the baseline and tuned model results
comparison = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1-score"
    ],
    "Task 2 Logistic Regression (Baseline)": [
        0.6669,
        0.2710,
        0.4327,
        0.2821
    ],
    "Task 3 Tuned Logistic Regression": [
        tuned_accuracy,
        tuned_precision,
        tuned_recall,
        tuned_f1
    ]
})

# Calculate the change in each metric
comparison["Change"] = (
    comparison["Task 3 Tuned Logistic Regression"]
    - comparison["Task 2 Logistic Regression (Baseline)"]
)

# Display the comparison table
display(comparison.round(4))

,Metric,Task 2 Logistic Regression (Baseline),Task 3 Tuned Logistic Regression,Change
0,Accuracy,0.6669,0.6622,-0.0047
1,Macro Precision,0.2710,0.2723,0.0013
2,Macro Recall,0.4327,0.4527,0.0200
3,Macro F1-score,0.2821,0.2837,0.0016


## Task 3 — Model Improvement Findings

### Objective
Improve and validate the baseline Logistic Regression model using hyperparameter tuning and stratified cross-validation.

### Methodology
- Applied a preprocessing pipeline for numerical and categorical features.
- Used median imputation and standard scaling for numerical features.
- Used most-frequent imputation and one-hot encoding for categorical features.
- Performed GridSearchCV with 3-fold Stratified Cross-Validation.
- Optimized the Macro F1-score to account for class imbalance.
- Evaluated the selected model on the same held-out test set used for the baseline comparison.

### Results
The tuned Logistic Regression model achieved a Macro F1-score of **0.2837**, compared with **0.2821** for the baseline model. Macro Recall increased from **0.4327** to **0.4527**, while accuracy decreased from **0.6669** to **0.6622**.

### Conclusion
Hyperparameter tuning produced a small improvement in Macro F1-score and a larger improvement in Macro Recall. However, overall performance remains limited by the imbalanced target classes. Further experiments with alternative models, class-imbalance handling techniques, and feature engineering may be useful.

The tuned model should not be considered production-ready without further validation.
